#### **Generics in Pydantic**

Generics allow you to create a **reusable model whose data type can vary**. For example, an API might return:

* a `User`
* a `Product`
* a `string`
* a `list`

while keeping the same response structure.

In [2]:
from typing import Generic, TypeVar
from pydantic import BaseModel

T = TypeVar("T")

class Response(BaseModel, Generic[T]):
    success: bool
    data: T

class User(BaseModel):
    id: int
    name: str

class Product(BaseModel):
    id: int
    price: float

# Use `Response` with different types:
user_response = Response[User](
    success=True,
    data={
        "id": 1,
        "name": "Shourov"
    }
)

product_response = Response[Product](
    success=True,
    data={
        "id": 10,
        "price": 99.99
    }
)
print(user_response.data.name)
print(product_response.data.price)

Shourov
99.99


#### **Why use Generics?**

Without generics, you might create:

```python
class UserResponse(BaseModel):
    success: bool
    data: User

class ProductResponse(BaseModel):
    success: bool
    data: Product
```

With generics, you need only one reusable model:

```python
class Response(BaseModel, Generic[T]):
    success: bool
    data: T
```

#### **Generic with lists**

```python
class ListResponse(BaseModel, Generic[T]):
    items: list[T]
    total: int

response = ListResponse[User](
    items=[
        {"id": 1, "name": "Alice"},
        {"id": 2, "name": "Bob"},
    ],
    total=2
)
```

This is especially useful when building **generic API response models in FastAPI**.

---

#### **Recursive Models**

A recursive model is a model that **contains itself**, directly or indirectly. A classic example is a tree structure:

```text
Company
 ├── Engineering
 │    ├── Backend
 │    └── Frontend
 └── Marketing
```

Each node can contain more nodes.

In [5]:
from pydantic import BaseModel

class Node(BaseModel):
    name: str
    children: list["Node"] = []

tree = Node(
    name="Root",
    children=[
        Node(name="Child 1", children=[Node(name="Grandchild")]),
        Node(name="Child 2")
    ]
)
print(tree.name)
print(tree.children[0].name)
print(tree.children[0].children[0].name)

Root
Child 1
Grandchild


#### **Where are recursive models useful?**

* File/folder structures
* Organization hierarchies
* Category trees
* Comments/replies
* Menu structures
* ASTs
* JSON trees
* Dependency graphs

**Important note:** For mutable defaults, Pydantic safely handles model fields, but it is still good practice to understand `default_factory`:

```python
from pydantic import BaseModel, Field

class Node(BaseModel):
    name: str
    children: list["Node"] = Field(default_factory=list)
```

This makes the intent explicit.

---

#### **Pydantic Dataclasses**

Pydantic provides dataclasses that combine the familiar Python `dataclass` style with **Pydantic validation**. Normal Python dataclass:

In [6]:
from dataclasses import dataclass

@dataclass
class User:
    name: str
    age: int

In [7]:
# Pydantic dataclass:
from pydantic.dataclasses import dataclass

@dataclass
class User:
    name: str
    age: int

user1 = User(name="Alice", age=25)
user2 = User(name="Alice", age="25")
print(user2.age)

25


#### **When should you use Pydantic dataclasses?**

* Already prefer Python's `dataclass` syntax
* Need Pydantic validation
* Want lightweight structured objects
* Are integrating with existing dataclass-based code

---
#### **Pydantic Settings**

`pydantic-settings` is used to load and validate **application configuration** especially from environment variables and `.env` files. This is extremely useful in FastAPI applications. Install it:

```bash
pip install pydantic-settings
```
```python
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_name: str
    debug: bool = False
    port: int = 8000
```

Now environment variables can provide values:

```text
APP_NAME=My FastAPI App
DEBUG=true
PORT=9000
```

```python
settings = Settings()

print(settings.app_name)
print(settings.debug)
print(settings.port)
```

---

#### **Using a `.env` file**

Create `.env` with:

```env
APP_NAME=My FastAPI App
DEBUG=true
PORT=9000
DATABASE_URL=postgresql://localhost/mydb
```

Then:

```python
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    app_name: str
    debug: bool = False
    port: int = 8000
    database_url: str

    model_config = SettingsConfigDict(
        env_file=".env"
    )

settings = Settings()
print(settings.app_name)
print(settings.database_url)
```

Pydantic Settings will:

1. Read environment variables / `.env`
2. Match them to your fields
3. Convert types
4. Validate them
5. Raise a validation error if required configuration is missing or invalid

---